# **Parte 1 – Scraping de decretos de emergencia**

### Responsable: Lucia Pulido

Ejecutar este notebook desde la carpeta `assignment_2/`, con las librerías `selenium`, `pandas`, `requests` y `beautifulsoup4` instaladas y un navegador Chrome (o Edge). La temporada del grupo 13 es del **1 de enero al 31 de mayo de 2022**. Los CSV se guardan en `datos/`.

## 1. Importar librerías

In [1]:
import time
import re
import requests
import pandas as pd

from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

## 2. Revisar robots.txt

In [2]:
import requests

robots_url = "https://www.gob.pe/robots.txt"

respuesta_robots = requests.get(
    robots_url,
    headers={"User-Agent": "Mozilla/5.0"}
)

print(respuesta_robots.status_code)
print(respuesta_robots.text)

200
# See http://www.robotstxt.org/robotstxt.html for documentation on how to use the robots.txt file
#
# To ban all spiders from the entire site uncomment the next two lines:
User-agent: *
Disallow: /admin/
Disallow: /*?sheet=
Disallow: /*&sheet=

User-agent: AhrefsBot
Disallow: /

User-agent: GPTBot
Crawl-delay: 5
Disallow: /admin/
Disallow: /*?sheet=
Disallow: /*&sheet=

User-agent: OAI-SearchBot
Crawl-delay: 2
Disallow: /admin/
Disallow: /*?sheet=
Disallow: /*&sheet=

Sitemap: https://www.gob.pe/sitemaps/sitemap.xml.gz



#### Revisión de robots.txt

##### El archivo robots.txt de gob.pe establece restricciones para determinados robots de rastreo. Para el usuario general (User-agent: *) se prohíbe el acceso a /admin/ y a determinadas URLs que contienen los parámetros sheet. No se observa una prohibición general para la ruta /busquedas, por lo que esta ruta puede ser consultada.

##### Sin embargo, el archivo establece un Crawl-delay de 5 segundos para GPTBot y de 2 segundos para OAI-SearchBot. Aunque estas restricciones no correspondan directamente al navegador utilizado con Selenium, es recomendable realizar pausas entre las consultas para evitar enviar demasiadas solicitudes al servidor en poco tiempo y reducir la carga sobre el sitio.

Además, `AhrefsBot` tiene prohibido todo el sitio (`Disallow: /`). Se usan pausas de al menos 2 segundos entre páginas de resultados y 1 segundo entre consultas a normas.

## 3. Configuración de Selenium

In [3]:
import os

# Chrome por defecto. Se puede elegir Edge con SCRAPING_BROWSER=edge.
navegador = os.environ.get("SCRAPING_BROWSER", "chrome").lower()
if navegador not in {"chrome", "edge"}:
    raise ValueError("SCRAPING_BROWSER debe ser chrome o edge")
options = webdriver.EdgeOptions() if navegador == "edge" else webdriver.ChromeOptions()
if os.environ.get("SCRAPING_HEADLESS") == "1":
    options.add_argument("--headless=new")
options.add_argument("--window-size=1440,1000")
driver = (webdriver.Edge if navegador == "edge" else webdriver.Chrome)(options=options)
wait = WebDriverWait(driver, 40)
verificaciones = []


def leer_total(driver):
    """Lee el contador del buscador, independientemente de los article extraídos."""
    elementos = driver.find_elements(
        By.CSS_SELECTOR,
        'main[aria-label="Resultados de la búsqueda"] h2[role="status"] span'
    )
    for elemento in elementos:
        texto = elemento.text.strip()
        coincidencia = re.fullmatch(r"([\d.,\s]+)\s+Resultados?", texto, re.I)
        if coincidencia:
            return int(re.sub(r"\D", "", coincidencia.group(1)))
    return None


def extraer_mes(mes, url):
    time.sleep(2)
    driver.get(url)
    wait.until(lambda d: leer_total(d) is not None)
    total = leer_total(driver)
    filas = []
    paginas = 0
    paginas_vistas = set()

    while len(filas) < total:
        articles = wait.until(EC.presence_of_all_elements_located((By.TAG_NAME, "article")))
        firma = tuple(a.find_element(By.CSS_SELECTOR, "h4 a").get_attribute("href") for a in articles)
        if firma in paginas_vistas:
            raise RuntimeError(f"{mes}: se repitió una página; revisar paginación")
        paginas_vistas.add(firma)
        paginas += 1
        for article in articles:
            enlace_elemento = article.find_element(By.CSS_SELECTOR, "h4 a")
            filas.append({
                "mes": mes,
                "numero": enlace_elemento.text.strip(),
                "fecha": article.find_element(
                    By.XPATH, ".//span[starts-with(normalize-space(.), 'Publicado:')]"
                ).text.strip(),
                "titulo": article.find_element(By.CSS_SELECTOR, "p").text.strip(),
                "enlace": enlace_elemento.get_attribute("href"),
            })

        if len(filas) >= total:
            break
        siguientes = driver.find_elements(
            By.XPATH,
            "//nav[contains(@class, 'search-pagination')]//*[self::a or self::button]"
            "[contains(translate(@aria-label, 'SIGUIENTE', 'siguiente'), 'siguiente') "
            "or contains(translate(normalize-space(.), 'SIGUIENTE', 'siguiente'), 'siguiente') "
            "or @rel='next']"
        )
        siguientes = [e for e in siguientes if e.is_displayed() and e.is_enabled()
                      and e.get_attribute("aria-disabled") != "true"]
        if not siguientes:
            raise RuntimeError(f"{mes}: hay {total} resultados pero solo {len(filas)} extraídos; revisar navegación")
        anterior = articles[0]
        time.sleep(2)
        siguientes[0].click()
        wait.until(EC.staleness_of(anterior))
        wait.until(lambda d: leer_total(d) == total)

    verificaciones.append({"mes": mes, "resultados_totales": total,
                           "resultados_extraidos": len(filas), "paginas": paginas})
    assert len(filas) == total, f"{mes}: los resultados no coinciden"
    print(f"{mes}: {total} anunciados; {len(filas)} extraídos; {paginas} página(s)")
    return filas

## 4. Scraping — Enero 2022

In [4]:
url_enero = (
    "https://www.gob.pe/busquedas?"
    "contenido[]=normas"
    "&institucion[]=pcm"
    "&term=estado%20de%20emergencia%20precipitaciones"
    "&sort_by=recent"
    "&desde=01-01-2022"
    "&hasta=31-01-2022"
)

datos_enero = extraer_mes("enero", url_enero)

enero: 8 anunciados; 8 extraídos; 1 página(s)


#### Extraer datos - Enero 2022

In [5]:
df_enero = pd.DataFrame(datos_enero, columns=["mes", "numero", "fecha", "titulo", "enlace"])
print("Resultados extraídos:", len(df_enero))

Resultados extraídos: 8


#### Tabla de documentos obtenidos — Enero 2022

In [6]:
df_enero

,mes,numero,fecha,titulo,enlace
0,enero,Decreto Supremo N.° 011-2022-PCM,Publicado: 30 de enero de 2022,Decreto Supremo que modifica el numeral 14.2 d...,https://www.gob.pe/institucion/pcm/normas-lega...
1,enero,Decreto Supremo N.° 010-2022-PCM,Publicado: 29 de enero de 2022,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
2,enero,Decreto Supremo N.° 009-2022-PCM,Publicado: 28 de enero de 2022,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
3,enero,Decreto Supremo N.° 004-2022-PCM,Publicado: 25 de enero de 2022,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
4,enero,Decreto Supremo N.° 007-2022-PCM,Publicado: 18 de enero de 2022,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
5,enero,Decreto Supremo N.° 006-2022-PCM,Publicado: 18 de enero de 2022,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
6,enero,Decreto Supremo N.° 005-2022-PCM,Publicado: 16 de enero de 2022,Decreto Supremo que modifica el Decreto Suprem...,https://www.gob.pe/institucion/pcm/normas-lega...
7,enero,Decreto Supremo N.° 002-2022-PCM,Publicado: 6 de enero de 2022,Decreto Supremo que modifica el numeral 8.1 de...,https://www.gob.pe/institucion/pcm/normas-lega...


## 5. Scraping — Febrero 2022

In [7]:
url_febrero = (
    "https://www.gob.pe/busquedas?"
    "contenido[]=normas"
    "&institucion[]=pcm"
    "&term=estado%20de%20emergencia%20precipitaciones"
    "&sort_by=recent"
    "&desde=01-02-2022"
    "&hasta=28-02-2022"
)

datos_febrero = extraer_mes("febrero", url_febrero)

febrero: 5 anunciados; 5 extraídos; 1 página(s)


#### Extraer datos - Febrero 2022

In [8]:
df_febrero = pd.DataFrame(datos_febrero, columns=["mes", "numero", "fecha", "titulo", "enlace"])
print("Resultados extraídos:", len(df_febrero))

Resultados extraídos: 5


#### Tabla de documentos obtenidos — Febrero 2022

In [9]:
df_febrero

,mes,numero,fecha,titulo,enlace
0,febrero,Decreto Supremo N.° 016-2022-PCM,Publicado: 27 de febrero de 2022,Decreto Supremo que declara Estado de Emergenc...,https://www.gob.pe/institucion/pcm/normas-lega...
1,febrero,Decreto Supremo N.° 015-2022-PCM,Publicado: 12 de febrero de 2022,Decreto Supremo que modifica el Decreto Suprem...,https://www.gob.pe/institucion/pcm/normas-lega...
2,febrero,Decreto Supremo N.° 014-2022-PCM,Publicado: 10 de febrero de 2022,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
3,febrero,Decreto Supremo N.° 013-2022-PCM,Publicado: 10 de febrero de 2022,Prórroga del Estado de Emergencia en distritos...,https://www.gob.pe/institucion/pcm/normas-lega...
4,febrero,Decreto Supremo N.° 012-2022-PCM,Publicado: 2 de febrero de 2022,Declarar por termino de cuarenta y cinco (45) ...,https://www.gob.pe/institucion/pcm/normas-lega...


## 6. Scraping — Marzo 2022

In [10]:
url_marzo = (
    "https://www.gob.pe/busquedas?"
    "contenido[]=normas"
    "&institucion[]=pcm"
    "&term=estado%20de%20emergencia%20precipitaciones"
    "&sort_by=recent"
    "&desde=01-03-2022"
    "&hasta=31-03-2022"
)

datos_marzo = extraer_mes("marzo", url_marzo)

marzo: 5 anunciados; 5 extraídos; 1 página(s)


#### Extraer datos - Marzo 2022

In [11]:
df_marzo = pd.DataFrame(datos_marzo, columns=["mes", "numero", "fecha", "titulo", "enlace"])
print("Resultados extraídos:", len(df_marzo))

Resultados extraídos: 5


#### Tabla de documentos obtenidos — Marzo 2022

In [12]:
df_marzo

,mes,numero,fecha,titulo,enlace
0,marzo,Decreto Supremo N.° 029-2022-PCM,Publicado: 26 de marzo de 2022,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
1,marzo,Decreto Supremo N.° 028-2022-PCM,Publicado: 26 de marzo de 2022,Decreto Supremo que declara el Estado de Emerg...,https://www.gob.pe/institucion/pcm/normas-lega...
2,marzo,Decreto Supremo N.° 027-2022-PCM,Publicado: 26 de marzo de 2022,| Decreto Supremo que declara Estado de Emerge...,https://www.gob.pe/institucion/pcm/normas-lega...
3,marzo,Decreto Supremo N.° 030-2022-PCM,Publicado: 22 de marzo de 2022,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
4,marzo,Resolución de Secretaría de Gestión Pública N....,Publicado: 11 de marzo de 2022,N° 002-2021-PCM/ SGP y adecúan de manera progr...,https://www.gob.pe/institucion/pcm/normas-lega...


## 7. Scraping — Abril 2022

In [13]:
url_abril = (
    "https://www.gob.pe/busquedas?"
    "contenido[]=normas"
    "&institucion[]=pcm"
    "&term=estado%20de%20emergencia%20precipitaciones"
    "&sort_by=recent"
    "&desde=01-04-2022"
    "&hasta=30-04-2022"
)

datos_abril = extraer_mes("abril", url_abril)

abril: 4 anunciados; 4 extraídos; 1 página(s)


#### Extraer datos - Abril 2022

In [14]:
df_abril = pd.DataFrame(datos_abril, columns=["mes", "numero", "fecha", "titulo", "enlace"])
print("Resultados extraídos:", len(df_abril))

Resultados extraídos: 4


#### Tabla de documentos obtenidos — Abril 2022

In [15]:
df_abril

,mes,numero,fecha,titulo,enlace
0,abril,Decreto Supremo N.° 041-2022-PCM,Publicado: 24 de abril de 2022,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
1,abril,Decreto Supremo N.° 035-2022-PCM,Publicado: 7 de abril de 2022,Decreto Supremo que declara el Estado de Emerg...,https://www.gob.pe/institucion/pcm/normas-lega...
2,abril,Decreto Supremo N.° 034-2022-PCM,Publicado: 4 de abril de 2022,Decreto Supremo que prorroga el Estado de emer...,https://www.gob.pe/institucion/pcm/normas-lega...
3,abril,Decreto Supremo N.° 032-2022-PCM,Publicado: 1 de abril de 2022,Decreto Supremo que declara el Estado de Emerg...,https://www.gob.pe/institucion/pcm/normas-lega...


## 8. Scraping — Mayo 2022

In [16]:
url_mayo = (
    "https://www.gob.pe/busquedas?"
    "contenido[]=normas"
    "&institucion[]=pcm"
    "&term=estado%20de%20emergencia%20precipitaciones"
    "&sort_by=recent"
    "&desde=01-05-2022"
    "&hasta=31-05-2022"
)

datos_mayo = extraer_mes("mayo", url_mayo)

mayo: 7 anunciados; 7 extraídos; 1 página(s)


#### Extraer datos - Mayo 2022

In [17]:
df_mayo = pd.DataFrame(datos_mayo, columns=["mes", "numero", "fecha", "titulo", "enlace"])
print("Resultados extraídos:", len(df_mayo))

Resultados extraídos: 7


#### Tabla de documentos obtenidos — Mayo 2022

In [18]:
df_mayo

,mes,numero,fecha,titulo,enlace
0,mayo,Decreto Supremo N.° 060-2022-PCM,Publicado: 26 de mayo de 2022,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
1,mayo,Decreto Supremo N.° 059-2022-PCM,Publicado: 26 de mayo de 2022,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
2,mayo,Decreto Supremo N.° 058-2022-PCM,Publicado: 24 de mayo de 2022,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
3,mayo,Decreto Supremo N.° 057-2022-PCM,Publicado: 20 de mayo de 2022,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
4,mayo,Decreto Supremo N.° 056-2022-PCM,Publicado: 18 de mayo de 2022,Decreto Supremo que declara el Estado de Emerg...,https://www.gob.pe/institucion/pcm/normas-lega...
5,mayo,Decreto Supremo N.° 055-2022-PCM,Publicado: 18 de mayo de 2022,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
6,mayo,Decreto Supremo N.° 054-2022-PCM,Publicado: 18 de mayo de 2022,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...


## 9. Consolidación de resultados

In [19]:
df_todos = pd.concat(
    [df_enero, df_febrero, df_marzo, df_abril, df_mayo],
    ignore_index=True
)

print("Total de resultados:", len(df_todos))

Total de resultados: 29


### 9.1. Tabla conjunta de documentos — Enero a Mayo 2022

In [20]:
df_todos = pd.concat(
    [df_enero, df_febrero, df_marzo, df_abril, df_mayo],
    ignore_index=True
)

df_todos

,mes,numero,fecha,titulo,enlace
0,enero,Decreto Supremo N.° 011-2022-PCM,Publicado: 30 de enero de 2022,Decreto Supremo que modifica el numeral 14.2 d...,https://www.gob.pe/institucion/pcm/normas-lega...
1,enero,Decreto Supremo N.° 010-2022-PCM,Publicado: 29 de enero de 2022,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
2,enero,Decreto Supremo N.° 009-2022-PCM,Publicado: 28 de enero de 2022,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
3,enero,Decreto Supremo N.° 004-2022-PCM,Publicado: 25 de enero de 2022,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
4,enero,Decreto Supremo N.° 007-2022-PCM,Publicado: 18 de enero de 2022,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
5,enero,Decreto Supremo N.° 006-2022-PCM,Publicado: 18 de enero de 2022,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
6,enero,Decreto Supremo N.° 005-2022-PCM,Publicado: 16 de enero de 2022,Decreto Supremo que modifica el Decreto Suprem...,https://www.gob.pe/institucion/pcm/normas-lega...
7,enero,Decreto Supremo N.° 002-2022-PCM,Publicado: 6 de enero de 2022,Decreto Supremo que modifica el numeral 8.1 de...,https://www.gob.pe/institucion/pcm/normas-lega...
8,febrero,Decreto Supremo N.° 016-2022-PCM,Publicado: 27 de febrero de 2022,Decreto Supremo que declara Estado de Emergenc...,https://www.gob.pe/institucion/pcm/normas-lega...
9,febrero,Decreto Supremo N.° 015-2022-PCM,Publicado: 12 de febrero de 2022,Decreto Supremo que modifica el Decreto Suprem...,https://www.gob.pe/institucion/pcm/normas-lega...


## 10. Verificación de resultados por mes

In [21]:
tabla_verificacion = pd.DataFrame(verificaciones)
assert len(tabla_verificacion) == 5
assert (tabla_verificacion["resultados_totales"] == tabla_verificacion["resultados_extraidos"]).all()
assert tabla_verificacion["resultados_extraidos"].sum() == len(df_todos)
display(tabla_verificacion)
driver.quit()

,mes,resultados_totales,resultados_extraidos,paginas
0,enero,8,8,1
1,febrero,5,5,1
2,marzo,5,5,1
3,abril,4,4,1
4,mayo,7,7,1


El total anunciado se lee del contador **«N Resultados»** del buscador; no se calcula a partir de los elementos extraídos. La columna `paginas` registra cuántas páginas se recorrieron. Si hay más resultados que los visibles, se intenta avanzar a la siguiente página. El programa se detiene si no puede recuperar todos los resultados o si los conteos no coinciden. Esta verificación se hace **antes** de eliminar duplicados y filtrar instituciones.

## 11. Eliminación de duplicados

In [22]:
antes = len(df_todos)

df_todos = df_todos.drop_duplicates(
    subset="enlace"
).reset_index(drop=True)

despues = len(df_todos)

print("Antes:", antes)
print("Después:", despues)
print("Duplicados eliminados:", antes - despues)

Antes: 29
Después: 29
Duplicados eliminados: 0


## 12. Filtrado de normas de la PCM

In [23]:
antes_pcm = len(df_todos)

df_todos = df_todos[
    df_todos["enlace"].str.contains(
        "/institucion/pcm/",
        na=False
    )
].copy()

eliminados_pcm = antes_pcm - len(df_todos)

print("Documentos eliminados:", eliminados_pcm)
print("Documentos restantes:", len(df_todos))

Documentos eliminados: 0
Documentos restantes: 29


## 13. Obtención de títulos completos

In [24]:
headers = {"User-Agent": "Mozilla/5.0"}
titulos_completos = []

for enlace in df_todos["enlace"]:
    time.sleep(1)
    respuesta = requests.get(enlace, headers=headers, timeout=40)
    respuesta.raise_for_status()
    sopa = BeautifulSoup(respuesta.text, "html.parser")
    descripcion = sopa.find("div", class_="description")
    if descripcion is None or not descripcion.get_text(strip=True):
        raise ValueError(f"No se pudo extraer el título completo: {enlace}")
    titulos_completos.append(descripcion.get_text(" ", strip=True))

df_todos["titulo"] = titulos_completos
assert df_todos["titulo"].str.strip().ne("").all()
print("Títulos completos no vacíos:", len(titulos_completos))

Títulos completos no vacíos: 29


## 14. Identificación de normas sobre lluvias

In [25]:
df_todos["es_lluvia"] = df_todos["titulo"].str.lower().apply(
    lambda x: "lluvia" in x or "precipitaciones" in x
)

## 15. Clasificar tipo de norma

In [26]:
def clasificar_tipo(titulo):

    titulo = titulo.lower()

    if "prórroga" in titulo or "prorroga" in titulo:
        return "prorroga"

    elif "declara" in titulo:
        return "declara"

    else:
        return "otro"

df_todos["tipo"] = df_todos["titulo"].apply(
    clasificar_tipo
)

## 16. Clasificación del motivo

In [27]:
def clasificar_motivo(titulo):

    titulo = titulo.lower()

    if "peligro inminente" in titulo:
        return "peligro inminente"

    elif "impacto de daños" in titulo:
        return "impacto de daños"

    else:
        return "otro"

df_todos["motivo"] = df_todos["titulo"].apply(
    clasificar_motivo
)

## 17. Revisión de casos clasificados como "otro"

### Para revisar el tipo:

In [28]:
df_todos[df_todos["tipo"] == "otro"][
    ["numero", "titulo", "tipo"]
]

,numero,titulo,tipo
17,Resolución de Secretaría de Gestión Pública N....,Dejan sin efecto la Res. N° 002-2021-PCM/ SGP ...,otro


### Para revisar el motivo:

In [29]:
df_todos[df_todos["motivo"] == "otro"][
    ["numero", "titulo", "motivo"]
]

,numero,titulo,motivo
0,Decreto Supremo N.° 011-2022-PCM,Decreto Supremo que modifica el numeral 14.2 d...,otro
1,Decreto Supremo N.° 010-2022-PCM,Decreto Supremo que prorroga el Estado de Emer...,otro
3,Decreto Supremo N.° 004-2022-PCM,Decreto Supremo que prorroga el Estado de Emer...,otro
4,Decreto Supremo N.° 007-2022-PCM,Decreto Supremo que prorroga el Estado de Emer...,otro
5,Decreto Supremo N.° 006-2022-PCM,Decreto Supremo que prorroga el Estado de Emer...,otro
6,Decreto Supremo N.° 005-2022-PCM,Decreto Supremo que modifica el Decreto Suprem...,otro
7,Decreto Supremo N.° 002-2022-PCM,Decreto Supremo que modifica el numeral 8.1 de...,otro
8,Decreto Supremo N.° 016-2022-PCM,Decreto Supremo que declara Estado de Emergenc...,otro
9,Decreto Supremo N.° 015-2022-PCM,Decreto Supremo que modifica el Decreto Suprem...,otro
11,Decreto Supremo N.° 013-2022-PCM,Prórroga del Estado de Emergencia en distritos...,otro


**Revisión de «otro».** La única fila con `tipo = "otro"` es una resolución de Secretaría de Gestión Pública cuyo título empieza «Dejan sin efecto…»: no declara ni prorroga una emergencia, por lo que esa clasificación es adecuada. Las 22 filas con `motivo = "otro"` no contienen literalmente «peligro inminente» ni «impacto de daños»; no les asignamos un motivo que el título no informa. Entre los resultados hay emergencias de otros tipos, como las vinculadas a la COVID-19. Todas esas 22 filas quedan fuera del filtro de lluvias. La norma que finalmente se conserva tiene `tipo = "declara"` y `motivo = "impacto de daños"`. Estas categorías siguen las reglas de búsqueda literal del enunciado; no equivalen a una lectura jurídica de cada norma.

## 18. Filtrado de normas sobre lluvias

In [30]:
df_lluvias = df_todos[
    df_todos["es_lluvia"] == True
].copy()

print(
    "Cantidad de normas sobre lluvias:",
    len(df_lluvias)
)

df_lluvias.head(5)

Cantidad de normas sobre lluvias: 1


,mes,numero,fecha,titulo,enlace,es_lluvia,tipo,motivo
21,abril,Decreto Supremo N.° 032-2022-PCM,Publicado: 1 de abril de 2022,Decreto Supremo que declara el Estado de Emerg...,https://www.gob.pe/institucion/pcm/normas-lega...,True,declara,impacto de daños


## 19. Identificación de departamentos

In [31]:
departamentos = [
    "Amazonas",
    "Áncash",
    "Apurímac",
    "Arequipa",
    "Ayacucho",
    "Cajamarca",
    "Callao",
    "Cusco",
    "Huancavelica",
    "Huánuco",
    "Ica",
    "Junín",
    "La Libertad",
    "Lambayeque",
    "Lima",
    "Loreto",
    "Madre de Dios",
    "Moquegua",
    "Pasco",
    "Piura",
    "Puno",
    "San Martín",
    "Tacna",
    "Tumbes",
    "Ucayali"
]

#### Función :

In [32]:
def identificar_departamentos(titulo):

    encontrados = []

    for departamento in departamentos:

        patron = r"\b" + re.escape(departamento) + r"\b"

        if re.search(patron, titulo):
            encontrados.append(departamento)

    return encontrados

df_lluvias["departamentos"] = df_lluvias["titulo"].apply(
    identificar_departamentos
)

df_lluvias[
    ["numero", "departamentos"]
]

,numero,departamentos
21,Decreto Supremo N.° 032-2022-PCM,"[Amazonas, Ayacucho, Piura]"


**Cómo evitamos contar mal.** En nuestra temporada, el Decreto Supremo N.° 032-2022-PCM menciona **Amazonas, Ayacucho y Piura**: se registra una vez para cada uno de esos tres departamentos. Recorremos una lista de nombres únicos y añadimos cada departamento como máximo una vez por decreto, aunque el nombre aparezca repetido como provincia y departamento. Los límites de palabra (`\b`) y la búsqueda en el título original evitan que «Ica» coincida con el final de «Huancavelica». En el único decreto de lluvia recuperado no aparece ese caso particular; no lo presentamos como un caso observado en esta muestra.

## 20. Declaratorias y prórrogas por departamento

In [33]:
tabla_departamentos = []

for departamento in departamentos:

    declaratorias = 0
    prorrogas = 0

    for _, fila in df_lluvias.iterrows():

        if departamento in fila["departamentos"]:

            if fila["tipo"] == "declara":
                declaratorias += 1

            elif fila["tipo"] == "prorroga":
                prorrogas += 1

    tabla_departamentos.append({
        "departamento": departamento,
        "declaratorias": declaratorias,
        "prorrogas": prorrogas
    })

df_departamentos = pd.DataFrame(
    tabla_departamentos
)

df_departamentos

,departamento,declaratorias,prorrogas
0,Amazonas,1,0
1,Áncash,0,0
2,Apurímac,0,0
3,Arequipa,0,0
4,Ayacucho,1,0
5,Cajamarca,0,0
6,Callao,0,0
7,Cusco,0,0
8,Huancavelica,0,0
9,Huánuco,0,0


## 21. Exportación de resultados

In [34]:
from pathlib import Path

Path("datos").mkdir(exist_ok=True)

df_lluvias[
    ["numero", "fecha", "titulo", "tipo", "motivo", "enlace"]
].to_csv(
    "datos/decretos_lluvias.csv",
    index=False,
    encoding="utf-8-sig"
)

In [35]:
df_departamentos.to_csv(
    "datos/decretos_por_departamento.csv",
    index=False,
    encoding="utf-8-sig"
)

In [36]:
print("Archivos guardados correctamente.")

Archivos guardados correctamente.


In [37]:
# Verificamos los archivos que utilizará la Parte 3.
decretos_guardados = pd.read_csv("datos/decretos_lluvias.csv")
departamentos_guardados = pd.read_csv("datos/decretos_por_departamento.csv")
assert list(decretos_guardados.columns) == ["numero", "fecha", "titulo", "tipo", "motivo", "enlace"]
assert list(departamentos_guardados.columns) == ["departamento", "declaratorias", "prorrogas"]
assert len(departamentos_guardados) == 25
assert departamentos_guardados["departamento"].is_unique
assert set(departamentos_guardados["departamento"]) == set(departamentos)
assert departamentos_guardados.notna().all().all()
print("CSV de normas:", len(decretos_guardados), "filas")
print("CSV por departamento:", len(departamentos_guardados), "filas")

CSV de normas: 1 filas
CSV por departamento: 25 filas


In [38]:
print("Resultados por tipo y motivo en las normas recuperadas:")
display(df_todos.groupby(["es_lluvia", "tipo", "motivo"]).size().reset_index(name="cantidad"))
print("Normas de lluvia y departamentos identificados:")
display(df_lluvias[["numero", "titulo", "tipo", "motivo", "departamentos"]])

Resultados por tipo y motivo en las normas recuperadas:


,es_lluvia,tipo,motivo,cantidad
0,False,declara,impacto de daños,1
1,False,declara,otro,9
2,False,otro,otro,1
3,False,prorroga,impacto de daños,5
4,False,prorroga,otro,12
5,True,declara,impacto de daños,1


Normas de lluvia y departamentos identificados:


,numero,titulo,tipo,motivo,departamentos
21,Decreto Supremo N.° 032-2022-PCM,Decreto Supremo que declara el Estado de Emerg...,declara,impacto de daños,"[Amazonas, Ayacucho, Piura]"


## Alcance de los datos para la Parte 3

Con la URL y las fechas indicadas en la tarea, el buscador anuncia y el notebook recupera **8, 5, 5, 4 y 7 resultados**, de enero a mayo: **29 en total**, sin duplicados y todos con enlace de la PCM. En esta consulta cada mes tiene una sola página. La tabla de verificación anterior muestra ambos conteos por separado.

Se leyeron los 29 títulos completos, sin títulos vacíos. Solo uno contiene «lluvia» o «precipitaciones»: el **DS N.° 032-2022-PCM**, del 1 de abril de 2022. Por eso `datos/decretos_lluvias.csv` tiene una fila y `datos/decretos_por_departamento.csv` tiene 25: Amazonas, Ayacucho y Piura tienen una declaratoria cada uno; el resto tiene cero, y no se identificaron prórrogas de lluvia en esta muestra.

Esta coincidencia de conteos verifica la extracción del **buscador con esos filtros**, pero no garantiza que su índice incluya todos los decretos de lluvias existentes. Para la Parte 3, los ceros significan «ninguna declaratoria encontrada con este procedimiento», no una prueba de ausencia de emergencias. Los tres departamentos empatan en el primer lugar; es una muestra muy pequeña y esta limitación debe explicarse al interpretar los gráficos y la conclusión.